# GNN + Temporal (Graph WaveNet-style) — Travel Time Model

A fifth model, and the one that actually changes the *unit of analysis*: instead of treating
each trip as an isolated sequence, this notebook builds a real network graph (stops as nodes,
observed stop-to-stop hops as edges — pulled straight from `build_gtfs_network.ipynb`'s
`segment_network.parquet`) and a **Graph WaveNet–style spatiotemporal encoder** (Wu et al., 2019)
that learns a "current network state" embedding for every stop, at every point in time, using
*all* trips/routes that passed through it — not just the one trip you're predicting.

## Why this is the biggest architectural leap of the five models
LSTM / Conv-Attn-LSTM / TFT / the non-causal Transformer all condition purely on one trip's own
history. None of them can express "M1 and M2 share three blocks of Broadway, so if M2 is running
slow right now, M1 is probably about to run slow too a few minutes later." That's a *cross-trip,
cross-route* spatial correlation, and it structurally cannot be captured by a per-trip recurrent
or attention model, no matter how good the encoder is — there's simply no channel for information
from a different trip to reach it. A graph over the shared stop network is the only way to give
the model that channel.

## Two-stage design (and why it's split this way, not end-to-end)
Running the full graph-time-series forward pass for every mini-batch of the per-trip head would
be extremely expensive (a graph convolution over `N` stops costs `O(N^2)` per timestep, and you'd
be paying that cost on every trip-batch, redundantly, since most trips share the same network
state at a given time). So this is trained in two clearly separate stages:

1. **Stage 1 — Graph WaveNet encoder**, trained as its own self-supervised forecasting task:
   given the recent history of aggregated real-time signals (delay, speed, headway, ridership) at
   every stop, predict the next time-bin's signals at every stop, network-wide. This is the
   standard formulation used in the traffic-forecasting literature (METR-LA / PEMS-BAY style
   sensor forecasting) and is a complete, legitimate SOTA-style model in its own right.
2. **Stage 2 — per-trip LSTM head**, structurally identical to `train_lstm.ipynb`'s baseline,
   but with one extra input per stop: a frozen embedding pulled from Stage 1's trained encoder,
   describing "what was the network's recent state at this stop, right before this trip got
   there." This makes the final comparison to the other four notebooks direct (same loss, same
   metrics, same trip-level split).

This is an approximation of true end-to-end joint training (Stage 1's weights are frozen when
Stage 2 trains) — a documented, common compromise for tractability. A natural extension, if you
want to push further for the paper, is unfreezing Stage 1 and fine-tuning both jointly at a low
learning rate.

## Causality — and why, unlike the Transformer notebook, this one *is* real-time-valid
Every piece here is deliberately causal: the graph encoder is built from dilated **causal**
convolutions (a stop's embedding at time *t* only ever depends on network signals from before
*t*), and Stage 2 looks up the embedding from the bin *before* the stop's own bin — so a trip's
own real-time observation can never leak into its own prediction. Unlike the non-causal
Transformer notebook, the numbers here should be a fair, live-ETA-comparable measurement, not an
offline upper bound. (Your stated aim wasn't live ETA, but this model happens to be valid for it
anyway, which is worth noting for the write-up.)

## Known limitations to flag in a research write-up
- **Graph is stop-hop adjacency, not physical road adjacency.** An edge exists only where GTFS
  stop sequences already show consecutive stops (`stop_id -> next_stop_id`) — including across
  different routes when they happen to share that exact pair. Two routes running on the same
  street but with different stop patterns (e.g. one stops more often) won't be connected. A
  proximity-based graph built from `shapes.parquet` geometry would capture more physical
  road-sharing but is real additional engineering, left as a documented extension.
- **Dense adjacency.** Both static supports and the learned adaptive adjacency are dense `N x N`
  matrices for implementation simplicity. Fine for a moderate stop count; for a much larger
  network you'd want sparse ops or a top-k-pruned adaptive adjacency.
- **Network-state aggregation mixes train/test trips.** Stage 1's per-bin aggregates are computed
  only from `train_df` rows, but the bin *index range* spans train+test so Stage 2 can look up
  embeddings for both. This is realistic for deployment (a live system always sees every bus
  currently running, not just "training" buses), but it does mean this isn't as strict a temporal
  holdout as a fully time-based train/test split would give you. Worth a robustness check with a
  by-date split if this becomes a paper claim.
- **Frozen embeddings, not end-to-end** (see above).

## Assumptions about your files (adjust the config cell if these don't match)
- `processed/segment_network.parquet` — from `build_gtfs_network.ipynb`, with `stop_id` /
  `next_stop_id` columns.
- `raw/processed_gtfs/baseline_dataset.parquet` — same file the other four notebooks use, assumed
  to also contain a `stop_id` column (per-row, identifying which physical stop that row's arrival
  is at) and a parseable `start_date`.


In [11]:
import math
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

# =====================================================================
# 0. Config
# =====================================================================
GRAPH_PATH = "processed/segment_network.parquet"
DATA_PATH = "raw/processed_gtfs/baseline_dataset.parquet"
TARGET = "travel_time"
SEQ_ID_COLS = ["trip_id", "start_date"]
ORDER_COL = "stop_sequence"
STOP_ID_COL = "stop_id"          # adjust if your baseline_dataset uses a different column name

CATEGORICAL = ["route_id", "direction_id", "shape_id", "service_id", 'is_raining', 'is_snowing', 'is_fog', "is_weekend",
    "is_federal_holiday", "is_school_day", "has_major_event", "is_peak", 'weathercode']
NUMERIC = [
    "stop_sequence", "trip_progress", "hour", "weekday", "month",
    "scheduled_arrival", "scheduled_departure", "scheduled_segment_time",
    "stop_lat", "stop_lon", "latitude", "longitude", "bearing",
    'temperature_c', 'precipitation_mm', 'snowfall_cm', 'windspeed_kmh',
    'segment_length', 'scheduled_segment_speed_mps', 'upstream_delay_seconds',
    'speed_mps', 'headway_seconds', "ridership", "transfers",
]
EMB_DIM_CAP = 150

# --- Stage 1 (Graph WaveNet encoder) config ---
SIGNAL_COLS = ["upstream_delay_seconds", "speed_mps", "headway_seconds", "ridership"]  # network-state signals
BIN_SECONDS = 1800          # 30-minute time bins
WINDOW = 12                 # history length fed to the encoder (12 bins = 6 hours)
GRAPH_HIDDEN = 32           # channel width inside the Graph WaveNet blocks
NODE_EMB_DIM = 16           # dim of the learned adaptive-adjacency node embeddings
DILATIONS = [1, 2, 4]       # one GraphWaveNetBlock per dilation
KERNEL_SIZE = 2
GRAPH_DROPOUT = 0.1
GRAPH_BATCH_SIZE = 32
GRAPH_LR = 1e-3
GRAPH_MAX_EPOCHS = 30
GRAPH_PATIENCE = 6

# --- Stage 2 (per-trip LSTM head) config — same defaults as train_lstm.ipynb ---
HIDDEN_SIZE = 256
NUM_LAYERS = 2
DROPOUT = 0.1
BATCH_SIZE = 64
LR = 1e-3
MAX_EPOCHS = 100
PATIENCE = 12

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)


## 1. Build the stop network graph from `segment_network.parquet`

In [12]:
segment_network = pl.read_parquet(GRAPH_PATH)
print(f"segment_network: {segment_network.shape}")

node_ids = (
    pl.concat([segment_network["stop_id"], segment_network["next_stop_id"]])
    .unique().sort().to_list()
)
node_index = {sid: i for i, sid in enumerate(node_ids)}
N_NODES = len(node_ids)
UNKNOWN_NODE_IDX = N_NODES   # extra row appended to the embedding table for unmapped stops
print(f"nodes (stops): {N_NODES:,}")

edges = (
    segment_network.select(["stop_id", "next_stop_id"]).unique().to_numpy()
)
src_idx = np.array([node_index[s] for s in edges[:, 0]])
dst_idx = np.array([node_index[s] for s in edges[:, 1]])
print(f"directed edges: {len(edges):,}")

# A[dst, src] = 1 where a bus travels src -> dst (row i lists i's upstream sources)
A = torch.zeros(N_NODES, N_NODES, dtype=torch.float32)
A[dst_idx, src_idx] = 1.0

def row_normalize(mat):
    row_sum = mat.sum(dim=1, keepdim=True).clamp(min=1e-8)
    return mat / row_sum

I = torch.eye(N_NODES)
support_fwd = row_normalize(A + I).to(DEVICE)      # "what flows into me" (Graph WaveNet's forward transition matrix)
support_bwd = row_normalize(A.t() + I).to(DEVICE)  # "what I flow into" (backward transition matrix)
print(f"static supports built: {support_fwd.shape}")


segment_network: (3976, 18)
nodes (stops): 585
directed edges: 679
static supports built: torch.Size([585, 585])


## 2. Load trip data, split, and build the time-binned network-state grid

In [ ]:
df = pl.read_parquet(DATA_PATH)
print(f"Loaded {df.shape}")

df = df.with_columns(
    (pl.col("trip_id") + "_" + pl.col("start_date").cast(pl.Utf8)).alias("_seq_id")
)

assert STOP_ID_COL in df.columns, (
    f"'{STOP_ID_COL}' not found in {DATA_PATH}. Set STOP_ID_COL in the config cell to the column "
    f"that identifies which physical stop each row is at. Available columns: {df.columns}"
)

groups = df["_seq_id"].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))
train_df = df[train_idx]
test_df = df[test_idx]
print(f"train rows: {train_df.height:,} | test rows: {test_df.height:,}")

# --- absolute event time -> bin index (assumes start_date parses as an ISO date; adjust if not) ---
# global_min_date = df.select(pl.col("start_date").cast(pl.Date, strict=False).min()).item()
df = df.with_columns(
    pl.col("start_date")
      .str.to_date("%Y%m%d")
      .alias("start_date")
)

global_min_date = df["start_date"].min()

def add_bin_index(d: pl.DataFrame) -> pl.DataFrame:

    day_offset = (
        pl.col("start_date") - pl.lit(global_min_date)
    ).dt.total_days()

    event_seconds = (
        pl.col("scheduled_arrival")
        + pl.col("upstream_delay_seconds").fill_null(0)
    )

    global_time = day_offset * 86400 + event_seconds

    return d.with_columns(
        (global_time // BIN_SECONDS)
        .cast(pl.Int64)
        .alias("_bin_idx")
    )

print(df.schema)
print(df.select(pl.col("start_date").head()))



Loaded (5596806, 41)
train rows: 4,479,933 | test rows: 1,116,873
Schema({'trip_id': String, 'start_date': Date, 'stop_id': String, 'route_id': String, 'direction_id': Int64, 'shape_id': String, 'service_id': String, 'stop_sequence': UInt32, 'trip_progress': Float64, 'hour': Int8, 'weekday': Int8, 'month': Int8, 'is_peak': Int8, 'scheduled_arrival': Int32, 'scheduled_departure': Int32, 'scheduled_segment_time': Int32, 'stop_lat': Float64, 'stop_lon': Float64, 'latitude': Float32, 'longitude': Float32, 'bearing': Float32, 'temperature_c': Float64, 'precipitation_mm': Float64, 'snowfall_cm': Float64, 'windspeed_kmh': Float64, 'is_raining': Int8, 'is_snowing': Int8, 'is_fog': Int8, 'weathercode': Int64, 'segment_length': Float64, 'scheduled_segment_speed_mps': Float64, 'upstream_delay_seconds': Int64, 'speed_mps': Float64, 'headway_seconds': Int64, 'is_weekend': Boolean, 'is_federal_holiday': Boolean, 'is_school_day': Boolean, 'has_major_event': Boolean, 'ridership': Int64, 'transfers': I

In [14]:
train_df = add_bin_index(train_df)
test_df = add_bin_index(test_df)

min_bin = min(train_df["_bin_idx"].min(), test_df["_bin_idx"].min())
max_bin = max(train_df["_bin_idx"].max(), test_df["_bin_idx"].max())
N_BINS = int(max_bin - min_bin) + 1
train_df = train_df.with_columns((pl.col("_bin_idx") - min_bin).alias("_bin_idx"))
test_df = test_df.with_columns((pl.col("_bin_idx") - min_bin).alias("_bin_idx"))
print(f"bin range: {N_BINS:,} bins of {BIN_SECONDS}s ({N_BINS * BIN_SECONDS / 86400:.1f} days)")


InvalidOperationError: - not allowed on str and date

In [ ]:
# =====================================================================
# 3. Aggregate TRAIN-only signals into a (bin, node, signal) grid
# =====================================================================
train_df = train_df.with_columns(
    pl.col(STOP_ID_COL).replace_strict(node_index, default=UNKNOWN_NODE_IDX).alias("_node_idx")
)

agg = (
    train_df.group_by(["_bin_idx", "_node_idx"])
    .agg([pl.col(c).mean().alias(c) for c in SIGNAL_COLS])
)

# fit normalization for the signal channels on observed TRAIN aggregates only
signal_means = {c: agg[c].mean() for c in SIGNAL_COLS}
signal_stds = {c: (agg[c].std() or 1.0) for c in SIGNAL_COLS}
signal_stds = {c: (s if s and s > 1e-8 else 1.0) for c, s in signal_stds.items()}

grid_values = np.full((N_BINS, N_NODES + 1, len(SIGNAL_COLS)), np.nan, dtype=np.float32)
grid_mask = np.zeros((N_BINS, N_NODES + 1, len(SIGNAL_COLS)), dtype=np.float32)

bins_np = agg["_bin_idx"].to_numpy()
nodes_np = agg["_node_idx"].to_numpy()
for i, c in enumerate(SIGNAL_COLS):
    vals = ((agg[c].to_numpy() - signal_means[c]) / signal_stds[c]).astype(np.float32)
    grid_values[bins_np, nodes_np, i] = vals
    grid_mask[bins_np, nodes_np, i] = 1.0

print(f"raw grid: {grid_values.shape}, observed fraction: {grid_mask.mean():.3f}")

# forward-fill missing (bin, node) entries along the bin axis, then fill any remaining
# leading gaps with 0 (already-normalized mean)
grid_filled = grid_values.copy()
last_seen = np.zeros((N_NODES + 1, len(SIGNAL_COLS)), dtype=np.float32)
have_seen = np.zeros((N_NODES + 1, len(SIGNAL_COLS)), dtype=bool)
for t in range(N_BINS):
    observed_t = grid_mask[t] > 0
    last_seen[observed_t] = grid_filled[t][observed_t]
    have_seen |= observed_t
    missing_t = ~observed_t
    grid_filled[t][missing_t & have_seen] = last_seen[missing_t & have_seen]
    # positions never yet observed stay at 0.0 (already the normalized mean)

# append an "was this bin genuinely observed" mask channel as an extra input feature
grid_input = np.concatenate([grid_filled, grid_mask.max(axis=-1, keepdims=True)], axis=-1)  # (N_BINS, N_NODES+1, C_in)
C_IN = grid_input.shape[-1]
C_OUT = len(SIGNAL_COLS)
print(f"model input grid: {grid_input.shape} (C_in={C_IN}, C_out={C_OUT})")

grid_input_t = torch.tensor(grid_input, dtype=torch.float32)
grid_target_mask_t = torch.tensor(grid_mask, dtype=torch.float32)   # for masking the forecasting loss


## 4. Graph WaveNet blocks — gated causal temporal conv + multi-support graph conv

Three supports are combined every block: the static forward transition matrix, the static
backward transition matrix, and a learned **adaptive adjacency** (Wu et al., 2019's signature
contribution) built from two small trainable node-embedding matrices — letting the model discover
correlations between stops that aren't in the schedule-derived graph at all.

In [ ]:
class GraphWaveNetBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation, n_supports, dropout):
        super().__init__()
        self.kernel_size = kernel_size
        self.dilation = dilation
        pad = (kernel_size - 1) * dilation
        self.filter_conv = nn.Conv1d(channels, channels, kernel_size, dilation=dilation, padding=pad)
        self.gate_conv = nn.Conv1d(channels, channels, kernel_size, dilation=dilation, padding=pad)
        self.graph_proj = nn.ModuleList([nn.Linear(channels, channels) for _ in range(n_supports)])
        self.skip_proj = nn.Linear(channels, channels)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(channels)

    def _causal_trim(self, x):
        trim = (self.kernel_size - 1) * self.dilation
        return x[:, :, :-trim] if trim > 0 else x

    def forward(self, x, supports):
        # x: (B, T, N, C)
        B, T, N, C = x.shape
        x_bn = x.permute(0, 2, 3, 1).reshape(B * N, C, T)          # (B*N, C, T) for Conv1d
        f = self._causal_trim(self.filter_conv(x_bn))
        g = self._causal_trim(self.gate_conv(x_bn))
        gated = torch.tanh(f) * torch.sigmoid(g)                    # WaveNet gated activation
        gated = gated.reshape(B, N, C, T).permute(0, 3, 1, 2)        # back to (B, T, N, C)
        gated = self.dropout(gated)

        graph_out = 0.0
        for proj, support in zip(self.graph_proj, supports):
            agg = torch.einsum("nm,btmc->btnc", support, gated)     # aggregate neighbor features
            graph_out = graph_out + proj(agg)

        out = self.norm(x + graph_out)      # residual + norm
        skip = self.skip_proj(out)
        return out, skip


class GraphWaveNetEncoder(nn.Module):
    def __init__(self, n_nodes, n_in_channels, hidden_channels, n_out_channels,
                 dilations, kernel_size, node_emb_dim, dropout):
        super().__init__()
        self.n_nodes = n_nodes
        self.input_proj = nn.Linear(n_in_channels, hidden_channels)
        self.node_emb1 = nn.Parameter(torch.randn(n_nodes, node_emb_dim) * 0.1)
        self.node_emb2 = nn.Parameter(torch.randn(n_nodes, node_emb_dim) * 0.1)
        n_supports = 3  # forward, backward, adaptive
        self.blocks = nn.ModuleList([
            GraphWaveNetBlock(hidden_channels, kernel_size, d, n_supports, dropout) for d in dilations
        ])
        self.output_layers = nn.Sequential(
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, n_out_channels),
        )

    def adaptive_support(self):
        logits = torch.relu(self.node_emb1 @ self.node_emb2.t())
        return torch.softmax(logits, dim=-1)

    def forward(self, x, support_fwd, support_bwd):
        # x: (B, T, N, C_in) -- N here must equal n_nodes (mapped/known nodes only; unknown-node
        # row is appended separately at lookup time in Stage 2, not part of this graph).
        h = self.input_proj(x)
        supports = [support_fwd, support_bwd, self.adaptive_support()]
        skip_sum = 0.0
        for block in self.blocks:
            h, skip = block(h, supports)
            skip_sum = skip_sum + skip
        pred = self.output_layers(skip_sum)   # (B, T, N, C_out): pred[:, t] estimates bin t+1
        return pred, skip_sum


## 5. Stage 1 training — windowed next-bin forecasting

In [ ]:
class GridWindowDataset(Dataset):
    def __init__(self, values, target_mask, window, known_nodes):
        self.values = values[:, known_nodes, :]            # drop the unknown-node row for training
        self.target_mask = target_mask[:, known_nodes, :]
        self.window = window

    def __len__(self):
        return self.values.shape[0] - self.window - 1

    def __getitem__(self, idx):
        x = self.values[idx: idx + self.window]                          # (window, N, C_in)
        y = self.values[idx + self.window, :, :C_OUT]                    # (N, C_out) next bin
        m = self.target_mask[idx + self.window]                          # (N, C_out)
        return x, y, m


known_node_slice = slice(0, N_NODES)   # exclude the appended unknown-node row for graph training
full_dataset = GridWindowDataset(grid_input_t, grid_target_mask_t, WINDOW, known_node_slice)

n_windows = len(full_dataset)
val_start = int(n_windows * 0.85)   # time-based split: last 15% of windows held out (not random)
train_windows = list(range(0, val_start))
val_windows = list(range(val_start, n_windows))
print(f"stage-1 windows: {n_windows:,} (train {len(train_windows):,} / val {len(val_windows):,})")

graph_train_loader = DataLoader(torch.utils.data.Subset(full_dataset, train_windows),
                                 batch_size=GRAPH_BATCH_SIZE, shuffle=True)
graph_val_loader = DataLoader(torch.utils.data.Subset(full_dataset, val_windows),
                               batch_size=GRAPH_BATCH_SIZE, shuffle=False)

encoder = GraphWaveNetEncoder(N_NODES, C_IN, GRAPH_HIDDEN, C_OUT, DILATIONS, KERNEL_SIZE,
                               NODE_EMB_DIM, GRAPH_DROPOUT).to(DEVICE)
graph_optimizer = torch.optim.AdamW(encoder.parameters(), lr=GRAPH_LR, weight_decay=1e-4)
n_params = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
print(f"Stage-1 trainable parameters: {n_params:,}")


def masked_mse_grid(pred, target, mask):
    return ((pred - target) ** 2 * mask).sum() / mask.sum().clamp(min=1)


def run_graph_epoch(loader, train: bool) -> float:
    encoder.train() if train else encoder.eval()
    total_loss, total_count = 0.0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for x, y, m in loader:
            x, y, m = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
            pred, _ = encoder(x, support_fwd, support_bwd)
            pred_last = pred[:, -1]   # forecast for the bin right after the window
            loss = masked_mse_grid(pred_last, y, m)
            if train:
                graph_optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=5.0)
                graph_optimizer.step()
            total_loss += loss.item() * x.size(0)
            total_count += x.size(0)
    return total_loss / max(total_count, 1)


best_val = float("inf")
no_improve = 0
best_encoder_state = None
for epoch in range(1, GRAPH_MAX_EPOCHS + 1):
    tr = run_graph_epoch(graph_train_loader, train=True)
    va = run_graph_epoch(graph_val_loader, train=False)
    print(f"[stage 1] epoch {epoch:3d} | train MSE {tr:.4f} | val MSE {va:.4f}")
    if va < best_val - 1e-5:
        best_val = va
        no_improve = 0
        best_encoder_state = {k: v.detach().cpu().clone() for k, v in encoder.state_dict().items()}
    else:
        no_improve += 1
        if no_improve >= GRAPH_PATIENCE:
            print(f"[stage 1] early stopping at epoch {epoch} (best val MSE {best_val:.4f})")
            break

encoder.load_state_dict(best_encoder_state)


## 6. Extract a frozen (bin, node) embedding table

One forward pass over the *entire* bin range (causal convolutions handle arbitrary sequence
length natively — no windowing needed at inference). This can be slow for a very long bin range
or a large stop network; reduce `GRAPH_HIDDEN`, use coarser `BIN_SECONDS`, or chunk this pass if
it doesn't fit in memory/time.

In [ ]:
encoder.eval()
with torch.no_grad():
    full_input = grid_input_t[:, :N_NODES, :].unsqueeze(0).to(DEVICE)      # (1, N_BINS, N_NODES, C_in)
    _, skip_sum_full = encoder(full_input, support_fwd, support_bwd)        # (1, N_BINS, N_NODES, hidden)
    embedding_table = skip_sum_full.squeeze(0).cpu()                        # (N_BINS, N_NODES, hidden)
    # append a zero row for the "unknown node" fallback
    embedding_table = torch.cat(
        [embedding_table, torch.zeros(N_BINS, 1, GRAPH_HIDDEN)], dim=1
    )                                                                        # (N_BINS, N_NODES+1, hidden)

print(f"embedding table: {embedding_table.shape}")


## 7. Stage 2 — per-trip LSTM head with graph-state features injected

From here this mirrors `train_lstm.ipynb` almost exactly: same categorical/numeric pipeline, same
model class, same loss and evaluation. The only change is that `NUMERIC` gets extended with the
graph embedding columns before normalization, so every downstream step (`build_sequences`,
`Dataset`, `LSTMTravelTime`) works unmodified.

In [ ]:
# lookup index for each row: embedding as of the bin *before* this row's own bin (causal —
# excludes this row's own contribution to the network-state grid)
def add_graph_embeddings(d: pl.DataFrame) -> pl.DataFrame:
    node_idx = d[STOP_ID_COL].replace_strict(node_index, default=UNKNOWN_NODE_IDX).to_numpy()
    lookup_bin = np.clip(d["_bin_idx"].to_numpy() - 1, 0, N_BINS - 1)
    emb = embedding_table[lookup_bin, node_idx, :].numpy()          # (n_rows, GRAPH_HIDDEN)
    emb_cols = {f"_graph_emb_{i}": emb[:, i].astype(np.float32) for i in range(GRAPH_HIDDEN)}
    return d.with_columns([pl.Series(name, vals) for name, vals in emb_cols.items()])


test_df = test_df.with_columns(
    pl.col(STOP_ID_COL).replace_strict(node_index, default=UNKNOWN_NODE_IDX).alias("_node_idx")
)
train_df = add_graph_embeddings(train_df)
test_df = add_graph_embeddings(test_df)

GRAPH_EMB_COLS = [f"_graph_emb_{i}" for i in range(GRAPH_HIDDEN)]
NUMERIC = NUMERIC + GRAPH_EMB_COLS   # extend the feature list used by every step below
print(f"NUMERIC now has {len(NUMERIC)} columns ({len(GRAPH_EMB_COLS)} from the graph encoder)")


In [ ]:
# =====================================================================
# 8. Encode categoricals / normalize numerics — fit on TRAIN only (identical to train_lstm.ipynb)
# =====================================================================
cat_maps = {}
cat_cardinalities = []
for col in CATEGORICAL:
    uniques = train_df[col].unique().sort().to_list()
    cat_maps[col] = {v: i + 1 for i, v in enumerate(uniques)}
    cat_cardinalities.append(len(uniques) + 1)


def encode_categoricals(d: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for col in CATEGORICAL:
        mapping = cat_maps[col]
        exprs.append(
            pl.col(col).cast(pl.Utf8).replace_strict(mapping, default=0).cast(pl.Int64).alias(f"_{col}_code")
        )
    return d.with_columns(exprs)


train_df = encode_categoricals(train_df)
test_df = encode_categoricals(test_df)
CAT_CODE_COLS = [f"_{c}_code" for c in CATEGORICAL]

num_means = {c: train_df[c].mean() for c in NUMERIC}
num_stds = {c: (train_df[c].std() or 1.0) for c in NUMERIC}
num_stds = {c: (s if s and s > 1e-8 else 1.0) for c, s in num_stds.items()}


def normalize_numeric(d: pl.DataFrame) -> pl.DataFrame:
    return d.with_columns([((pl.col(c) - num_means[c]) / num_stds[c]).alias(c) for c in NUMERIC])


train_df = normalize_numeric(train_df)
test_df = normalize_numeric(test_df)


In [ ]:
# =====================================================================
# 9. Sequence building / Dataset / collate — identical pattern to train_lstm.ipynb
# =====================================================================
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence


def build_sequences(d: pl.DataFrame) -> list[dict]:
    d = d.sort(SEQ_ID_COLS + [ORDER_COL])
    sequences = []
    for _, group in d.group_by("_seq_id", maintain_order=True):
        cat = group.select(CAT_CODE_COLS).to_numpy()
        num = group.select(NUMERIC).to_numpy().astype(np.float32)
        target = group[TARGET].to_numpy().astype(np.float32)
        sequences.append({"cat": cat, "num": num, "target": target, "length": len(target)})
    return sequences


print("Building train sequences...")
train_sequences = build_sequences(train_df)
print("Building test sequences...")
test_sequences = build_sequences(test_df)
print(f"train sequences: {len(train_sequences):,} | test sequences: {len(test_sequences):,}")


class TripSequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        s = self.sequences[idx]
        return (
            torch.tensor(s["cat"], dtype=torch.long),
            torch.tensor(s["num"], dtype=torch.float32),
            torch.tensor(s["target"], dtype=torch.float32),
            s["length"],
        )


def collate(batch):
    cats, nums, targets, lengths = zip(*batch)
    lengths = torch.tensor(lengths, dtype=torch.long)
    return (
        pad_sequence(cats, batch_first=True, padding_value=0),
        pad_sequence(nums, batch_first=True, padding_value=0.0),
        pad_sequence(targets, batch_first=True, padding_value=0.0),
        lengths,
    )


train_loader = DataLoader(TripSequenceDataset(train_sequences), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
test_loader = DataLoader(TripSequenceDataset(test_sequences), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)


In [ ]:
# =====================================================================
# 10. Model — same LSTMTravelTime architecture as train_lstm.ipynb (graph info arrives purely
#     through the extended NUMERIC feature vector, so the head itself needs no changes)
# =====================================================================
class LSTMTravelTime(nn.Module):
    def __init__(self, cat_cardinalities, n_numeric, hidden_size, num_layers, dropout):
        super().__init__()
        emb_dims = [min(EMB_DIM_CAP, max(1, int(1.6 * (c ** 0.56)))) for c in cat_cardinalities]
        self.e = nn.ModuleList([nn.Embedding(c, d, padding_idx=0) for c, d in zip(cat_cardinalities, emb_dims)])
        in_dim = sum(emb_dims) + n_numeric
        self.l = nn.LSTM(in_dim, hidden_size, num_layers=num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.h = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, cat, num, lengths):
        T = num.size(1)
        emb = [e(cat[:, :, i]) for i, e in enumerate(self.e)]
        x = torch.cat(emb + [num], dim=-1)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.l(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=T)
        return self.h(out).squeeze(-1)


def masked_mae(pred, target, lengths):
    mask = (torch.arange(pred.size(1), device=pred.device).unsqueeze(0) < lengths.unsqueeze(1).to(pred.device))
    return (torch.abs(pred - target) * mask).sum() / mask.sum().clamp(min=1)


def masked_mse(pred, target, lengths):
    mask = (torch.arange(pred.size(1), device=pred.device).unsqueeze(0) < lengths.unsqueeze(1).to(pred.device))
    return (((pred - target) ** 2) * mask).sum() / mask.sum().clamp(min=1)


model = LSTMTravelTime(cat_cardinalities, len(NUMERIC), HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, epochs=MAX_EPOCHS, steps_per_epoch=len(train_loader))
print(model)
print(f"trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# =====================================================================
# 11. Train loop with early stopping (identical structure to train_lstm.ipynb)
# =====================================================================
def run_epoch(loader, train: bool) -> float:
    model.train() if train else model.eval()
    total_mae, total_count = 0.0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for cat, num, target, lengths in loader:
            cat, num, target = cat.to(DEVICE), num.to(DEVICE), target.to(DEVICE)
            pred = model(cat, num, lengths)
            loss = masked_mse(pred, target, lengths)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
                scheduler.step()
            mae = masked_mae(pred, target, lengths)
            n = lengths.sum().item()
            total_mae += mae.item() * n
            total_count += n
    return total_mae / total_count


best_val_mae = float("inf")
epochs_no_improve = 0
best_state = None
for epoch in range(1, MAX_EPOCHS + 1):
    train_mae = run_epoch(train_loader, train=True)
    val_mae = run_epoch(test_loader, train=False)
    print(f"epoch {epoch:3d} | train MAE {train_mae:7.2f}s | val MAE {val_mae:7.2f}s")
    if val_mae < best_val_mae - 1e-3:
        best_val_mae = val_mae
        epochs_no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best val MAE {best_val_mae:.2f}s)")
            break

model.load_state_dict(best_state)


In [ ]:
# =====================================================================
# 12. Final evaluation — same metrics as the other four notebooks
# =====================================================================
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for cat, num, target, lengths in test_loader:
        cat, num = cat.to(DEVICE), num.to(DEVICE)
        pred = model(cat, num, lengths).cpu()
        for i, length in enumerate(lengths):
            all_preds.append(pred[i, :length].numpy())
            all_targets.append(target[i, :length].numpy())

preds = np.concatenate(all_preds)
targets = np.concatenate(all_targets)
resid = preds - targets

mae = np.abs(resid).mean()
rmse = np.sqrt((resid ** 2).mean())
median_ae = np.median(np.abs(resid))
bias = resid.mean()
r2 = 1 - (resid ** 2).sum() / ((targets - targets.mean()) ** 2).sum()

print(f"\nMAE:       {mae:.1f} sec")
print(f"Median AE: {median_ae:.1f} sec")
print(f"RMSE:      {rmse:.1f} sec")
print(f"Bias:      {bias:+.1f} sec")
print(f"R2:        {r2:.4f}")
for thresh in (30, 60, 120):
    print(f"within {thresh}s: {(np.abs(resid) <= thresh).mean():.1%}")

import os
os.makedirs("models", exist_ok=True)
torch.save(
    {
        "model_state": model.state_dict(),
        "encoder_state": encoder.state_dict(),
        "node_index": node_index,
        "cat_maps": cat_maps, "num_means": num_means, "num_stds": num_stds,
        "signal_means": signal_means, "signal_stds": signal_stds,
        "config": {
            "HIDDEN_SIZE": HIDDEN_SIZE, "NUM_LAYERS": NUM_LAYERS, "DROPOUT": DROPOUT,
            "GRAPH_HIDDEN": GRAPH_HIDDEN, "BIN_SECONDS": BIN_SECONDS, "WINDOW": WINDOW,
            "SIGNAL_COLS": SIGNAL_COLS, "NUMERIC": NUMERIC, "CATEGORICAL": CATEGORICAL,
        },
    },
    "models/gnn_temporal_lstm.pt",
)
print("\nmodel saved to models/gnn_temporal_lstm.pt")
